<a href="https://colab.research.google.com/github/monicachet99/NLP-DL/blob/main/Q%26A_model_for_COI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd

!pip install kaggle -qq

dataset_identifier = "piyushsingh80768/constitution-of-india"

print(f"Downloading dataset: {dataset_identifier}...")
!kaggle datasets download -d {dataset_identifier} -q

downloaded_zip_file = dataset_identifier.split('/')[-1] + '.zip'
print(f"Downloaded file: {downloaded_zip_file}")

print(f"Unzipping {downloaded_zip_file}...")
!unzip -o {downloaded_zip_file} -d .

print("\nFiles extracted:")
!ls

file_to_load = '20240716890312078.pdf'

try:
    data = pd.read_csv(file_to_load)
    print(f"\nSuccessfully loaded '{file_to_load}' into 'data' variable.")
    print("\nFirst 5 rows of the data:")
    print(data.head())
    print(f"\nShape of the data: {data.shape}")
except FileNotFoundError:
    print(f"\nError: The file '{file_to_load}' was not found after unzipping.")
    print("Please check the 'Files extracted:' output above and adjust 'file_to_load' variable if necessary.")
except Exception as e:
    print(f"\nAn error occurred while loading the data: {e}")

In [ ]:
data.Articles[0]

In [ ]:
import re

article_pattern = r"^(\d+[A-Z]?\.)\s*(.*)"

data[['Article Number', 'Description']] = data['Articles'].str.extract(article_pattern, flags=re.DOTALL)

data = data.drop(columns=['Articles'])

print("DataFrame with 'Article Number' and 'Description' columns:")
display(data.head())

In [ ]:

!pip install -U sentence-transformers -qq
!pip install faiss-cpu -qq

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings for article descriptions...")
article_embeddings = model.encode(data['Description'].tolist(), show_progress_bar=True)

print(f"Generated {len(article_embeddings)} embeddings of dimension {article_embeddings.shape[1]}.")

dimension = article_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(article_embeddings)

print(f"FAISS index created with {index.ntotal} vectors.")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import faiss
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model_llm = AutoModelForCausalLM.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

print("Local LLM (TinyLlama) loaded successfully.")

def ask_constitution(query, k=3):
    query_embedding = model.encode([query])

    distances, indices = index.search(query_embedding, k)

    retrieved_articles = []
    for i, idx in enumerate(indices[0]):
        article_number = data.loc[idx, 'Article Number']
        description = data.loc[idx, 'Description']
        retrieved_articles.append(f"Article {article_number}: {description}")

    context = "\n\n".join(retrieved_articles)
    prompt_template = f"""You are a helpful assistant that answers questions about the Constitution of India.
    Use the following articles to answer the user's question. If the information is not present in the articles, state that you cannot find the answer in the provided context.

    Context Articles:
    {context}

    User Question: {query}

    Answer:"""

    try:
        inputs = tokenizer(prompt_template, return_tensors="pt", truncation=True, max_length=512)

        outputs = model_llm.generate(
            **inputs,
            max_new_tokens=250,
            num_return_sequences=1,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7,
            pad_token_id=tokenizer.eos_token_id
        )

        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        answer_start = generated_text.find("Answer:")
        if answer_start != -1:
            return generated_text[answer_start + len("Answer:"):].strip()
        else:
            return generated_text.strip()

    except Exception as e:
        return f"An error occurred while generating the response with the local model: {e}"

print("Q&A system function `ask_constitution` defined using a local LLM.")

In [ ]:
query = "What is the capital of India?"
response = ask_constitution(query)
print(response)

In [ ]:
query = "what are my fundamental rights"
response = ask_constitution(query)
print(response)

In [ ]:
query = "can i discriminate someone"
response = ask_constitution(query)
print(response)